# convtranspose-bn-activation-block — ex1: build a 4x4 to 8x8 generator upsampling block

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `convtranspose-bn-activation-block`. Running the final beacon cell reports progress against the `GAN: ConvT+BN+Activation block` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: ConvT+BN+Activation block` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`convtranspose-bn-activation-block`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "convtranspose-bn-activation-block"
DD_SUBTOPIC = "GAN: ConvT+BN+Activation block"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## ConvTranspose + BN + activation block — quick refresher

The repeated unit of a DCGAN generator. Three layers stacked in `nn.Sequential`:

```python
nn.Sequential(
    nn.ConvTranspose2d(in_c, out_c, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(out_c),
    nn.ReLU(inplace=True),
)
```

**Stride 2, kernel 4, padding 1 doubles spatial size.** Output shape `H_out = (H_in - 1) * stride - 2 * padding + kernel = 2 * H_in`. So `4 -> 8 -> 16 -> 32 -> 64` for a 4-block stack.

**`bias=False`** because BatchNorm immediately re-centres the feature map — the ConvTranspose bias is redundant and just wastes parameters. Standard DCGAN convention.

**ReLU (not LeakyReLU) on the GENERATOR.** Discriminator uses LeakyReLU; generator uses ReLU. The asymmetry is from the original paper — the generator needs sharp activations to produce crisp outputs, the discriminator needs a gentle slope on negatives to avoid dying neurons.

### Exercise 1 — build a 4x4 to 8x8 generator upsampling block

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `nn.Sequential` to wire `nn.ConvTranspose2d(stride=2, kernel=4, padding=1, bias=False) -> nn.BatchNorm2d -> nn.ReLU` into a single channel-halving, spatial-doubling generator block.
> Keywords: gan, generator, convtranspose, batchnorm, sequential
> ```

**KCs targeted:** `sequential-three-layer-block`, `convtranspose-stride-doubles-spatial`

Implement `ex1_build_generator_block(in_channels, out_channels)`. The repeated unit of a DCGAN generator:

1. Construct an `nn.Sequential` containing three layers IN ORDER:
   - `nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False)`
   - `nn.BatchNorm2d(out_channels)`
   - `nn.ReLU(inplace=True)`
2. `bias=False` on the ConvTranspose because BatchNorm immediately follows.
3. Stride 2 + kernel 4 + padding 1 DOUBLES the spatial size — a 4×4 input becomes 8×8 output.
4. Return the Sequential.

Input: `in_channels`, `out_channels` — ints.
Output: `nn.Sequential` module.

The visualization runs your block on a `(1, 1024, 4, 4)` seed and renders four output channel slices as a 2×2 grid of 8×8 feature maps.

In [ ]:
def ex1_build_generator_block(in_channels: int, out_channels: int) -> t.nn.Sequential:
    """Return nn.Sequential(ConvTranspose2d, BatchNorm2d, ReLU)."""
    raise NotImplementedError()


def _test_ex1():
    import torch.nn as nn

    # Build a 4x4 -> 8x8 generator block (channel halving).
    block = ex1_build_generator_block(in_channels=1024, out_channels=512)
    assert isinstance(block, nn.Sequential), 'must return nn.Sequential'
    layers = list(block.children())
    assert len(layers) == 3, f'expected exactly 3 layers, got {len(layers)}'
    assert isinstance(layers[0], nn.ConvTranspose2d), f'layer 0 must be ConvTranspose2d, got {type(layers[0]).__name__}'
    assert isinstance(layers[1], nn.BatchNorm2d), f'layer 1 must be BatchNorm2d, got {type(layers[1]).__name__}'
    assert isinstance(layers[2], nn.ReLU), f'layer 2 must be ReLU, got {type(layers[2]).__name__}'

    # ConvTranspose configuration.
    ct = layers[0]
    assert ct.in_channels == 1024 and ct.out_channels == 512, f'ConvT channels wrong: in={ct.in_channels}, out={ct.out_channels}'
    assert ct.kernel_size == (4, 4), f'kernel must be 4, got {ct.kernel_size}'
    assert ct.stride == (2, 2), f'stride must be 2, got {ct.stride}'
    assert ct.padding == (0, 0) or ct.padding == (1, 1), f'padding should be 1, got {ct.padding}'
    assert ct.padding == (1, 1), f'padding must be 1 for spatial-doubling, got {ct.padding}'
    assert ct.bias is None, 'bias must be False (BatchNorm follows)'

    # BatchNorm configuration.
    bn = layers[1]
    assert bn.num_features == 512, f'BatchNorm num_features must match ConvT out, got {bn.num_features}'

    # Shape behavior: (B, 1024, 4, 4) → (B, 512, 8, 8). Use eval() so BN is identity-ish.
    block.eval()
    with t.no_grad():
        seed = t.randn(2, 1024, 4, 4)
        out = block(seed)
        assert out.shape == (2, 512, 8, 8), f'expected (2,512,8,8), got {tuple(out.shape)}'
        # ReLU output → all nonnegative.
        assert (out >= 0).all(), 'ReLU activation must zero out negatives'

    # A second build with different channels to confirm parametrization.
    small = ex1_build_generator_block(in_channels=128, out_channels=64)
    small.eval()
    with t.no_grad():
        out_small = small(t.randn(1, 128, 16, 16))
        assert out_small.shape == (1, 64, 32, 32), f'expected (1,64,32,32), got {tuple(out_small.shape)}'

    # --- Visualization: 4 output channels as 8x8 feature maps ---
    block.eval()
    with t.no_grad():
        rng = t.Generator().manual_seed(0)
        seed_viz = t.randn(1, 1024, 4, 4, generator=rng)
        out_viz = block(seed_viz)
    fig, axes = plt.subplots(2, 2, figsize=(6, 6))
    for i, ax in enumerate(axes.flat):
        ax.imshow(out_viz[0, i].numpy(), cmap='viridis')
        ax.set_title(f'channel {i} (8×8)')
        ax.axis('off')
    plt.suptitle('ex1 generator block output — 4 channel slices (4×4 → 8×8)')
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_build_generator_block(in_channels: int, out_channels: int) -> t.nn.Sequential:
    import torch.nn as nn
    return nn.Sequential(
        nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )
```

**Why kernel=4, stride=2, padding=1.** This combination is the DCGAN paper standard. Output size for ConvTranspose2d is `H_out = (H_in - 1) * stride - 2 * padding + kernel = (H_in - 1) * 2 - 2 + 4 = 2 * H_in`. Exact doubling.

**Why ReLU, not LeakyReLU.** Generator uses ReLU; discriminator uses LeakyReLU. The DCGAN authors found this asymmetry empirically stabilizes training. Generator needs crisp activations to produce sharp images; discriminator needs a gentle negative slope to keep learning from fakes.

**`bias=False` matters.** Adding a bias to the ConvTranspose would be redundant since BatchNorm has its own `affine` parameters (`weight` and `bias`) and re-centres the entire feature map. Free parameter savings: `out_channels` per block.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()